# ATLAS YOLO26 Component Training

1. Export component dataset from the annotator (`⚡ Export Components`)
2. Upload to Google Drive at `My Drive/Atlas/train/`
3. Set **Runtime → Change runtime type → T4 GPU**
4. Run all cells top to bottom

**Classes**: 58 classes — component marks with phase variants (MCB 2/3 Phase, ELB 2/3 Phase, MC 2/3 Phase, THR 2/3 Phase, etc.)


In [ ]:
# ── CONFIGURATION ─────────────────────────────────────────────────────────────
# Path inside Google Drive to the YOLO dataset folder
# Must contain: data.yaml, train/images/, train/labels/, valid/images/, valid/labels/
DRIVE_FOLDER = "Atlas/train"

# Where to copy the dataset on the Colab VM (don't change)
DATASET_DIR  = "/content/atlas_dataset"
# ──────────────────────────────────────────────────────────────────────────────

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Copy dataset from Drive to local VM
import os, shutil

src = f"/content/drive/MyDrive/{DRIVE_FOLDER}"
if not os.path.exists(src):
    raise FileNotFoundError(
        f"Folder not found: {src}\n\n"
        f"Upload your YOLO dataset to Google Drive at:\n"
        f"  My Drive/{DRIVE_FOLDER}/\n\n"
        f"It should contain: data.yaml, train/, valid/"
    )

if os.path.exists(DATASET_DIR):
    shutil.rmtree(DATASET_DIR)
shutil.copytree(src, DATASET_DIR)
print(f"Copied: {src} → {DATASET_DIR}")
print("Contents:", os.listdir(DATASET_DIR))

In [ ]:
# 3. Verify dataset structure & print data.yaml
import os

yaml_path = os.path.join(DATASET_DIR, 'data.yaml')

if not os.path.exists(yaml_path):
    raise FileNotFoundError(
        "data.yaml not found!\n"
        "Export from annotator using '⚡ Export Components' button first."
    )

print("data.yaml:\n", open(yaml_path).read())

for split in ('train', 'valid'):
    imgs = os.path.join(DATASET_DIR, split, 'images')
    lbls = os.path.join(DATASET_DIR, split, 'labels')
    n_img = len(os.listdir(imgs)) if os.path.exists(imgs) else 0
    n_lbl = len(os.listdir(lbls)) if os.path.exists(lbls) else 0
    print(f"{split}: {n_img} images, {n_lbl} label files")

In [ ]:
# 4. Install Ultralytics (includes YOLO26)
!pip install ultralytics -q

In [ ]:
# 5. Fix data.yaml paths to absolute Colab paths
import os, yaml

with open(yaml_path) as f:
    data = yaml.safe_load(f)

data['path']  = DATASET_DIR
data['train'] = 'train/images'

# If no validation images were exported, reuse train for val
valid_imgs = os.path.join(DATASET_DIR, 'valid', 'images')
if not os.path.exists(valid_imgs) or len(os.listdir(valid_imgs)) == 0:
    print('⚠️  No validation images found — using train set for val')
    data['val'] = 'train/images'
else:
    data['val'] = 'valid/images'

with open(yaml_path, 'w') as f:
    yaml.dump(data, f)

print('Updated data.yaml:')
print(open(yaml_path).read())


In [ ]:
# 7. Train YOLO26n on component detection (classes = component marks)
from ultralytics import YOLO

model = YOLO('yolo26n.pt')  # nano: fastest training, good for iteration

results = model.train(
    data=yaml_path,
    epochs=150,
    imgsz=1280,       # large — schematics have small fine detail
    batch=16,         # nano is lightweight, can use larger batches
    patience=30,      # early stopping — more patience with many classes
    optimizer='AdamW',
    lr0=0.001,
    weight_decay=0.0005,
    augment=True,
    mosaic=0.5,
    flipud=0.0,       # schematics shouldn't be flipped vertically
    fliplr=0.0,       # or horizontally
    degrees=2.0,      # slight rotation ok
    project='/content/drive/MyDrive/atlas_runs',  # save to Drive
    name='yolo26n_components_v3',                 # v3: 18 pages, 58 classes, fully validated
    exist_ok=True,
)


In [ ]:
# 8. Validate
metrics = model.val()
print('mAP50:',    metrics.box.map50)
print('mAP50-95:', metrics.box.map)

In [ ]:
# 9. Run inference on ALL images with the just-trained model
import glob
from ultralytics import YOLO
from IPython.display import Image as IPImage, display

# Use the model we just trained (still in memory)
all_images = sorted(
    glob.glob(os.path.join(DATASET_DIR, 'train/images/*.*')) +
    glob.glob(os.path.join(DATASET_DIR, 'valid/images/*.*'))
)
print(f'Found {len(all_images)} images to test')

# Run inference on each image
for img_path in all_images:
    page_name = os.path.basename(img_path)
    print(f'\n--- {page_name} ---')
    results = model.predict(
        img_path,
        conf=0.25,
        save=True,
        project='/content/predictions',
        name='component_test',
        exist_ok=True,
    )
    # Show detection count per class
    for r in results:
        if r.boxes is not None and len(r.boxes):
            cls_ids = r.boxes.cls.int().tolist()
            names = r.names
            from collections import Counter
            counts = Counter(cls_ids)
            for cid, cnt in sorted(counts.items()):
                print(f'  {names[cid]}: {cnt}')
        else:
            print('  No detections')

# Display all predicted images
saved = sorted(glob.glob('/content/predictions/component_test/*.*'))
print(f'\n{len(saved)} prediction images saved')
for s in saved:
    print(f'\n{os.path.basename(s)}:')
    display(IPImage(s, width=900))

In [ ]:
# 10. Check best weights location
import os
best_weights = '/content/drive/MyDrive/atlas_runs/yolo26n_components_v3/weights/best.pt'
print('Best weights saved at:', best_weights)
print('Exists:', os.path.exists(best_weights))
